# Master Notebook
## Getting CFs based on user's preferences with LLM interface

In [1]:
# ============================================================
# 0) Libraries
# ============================================================
%env CUDA_VISIBLE_DEVICES=1

import torch
import json
import numpy as np
import pandas as pd
import dice_ml
import copy
import re

from typing import Any, Dict
from jsonschema import validate
from jsonschema.exceptions import ValidationError
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from catboost import CatBoostRegressor
from pathlib import Path
from dice_ml import Dice

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.7, device=0)
    print("CUDA available.")
else:
    print("CUDA not available. Running on CPU.")



env: CUDA_VISIBLE_DEVICES=1
CUDA available.


In [2]:
# ============================================================
# 1) Config
# ============================================================

CSV_PATH = Path("surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv")
# ART_DIR  = Path("surrogate_pkl_cfs_metadata")
MODEL_PATH  = Path("surrogate_pkl_cfs_metadata/surrogate_catboost.cbm")

TARGET = "value"

FEATURES = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_coreset_sampling_ratio",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_num_neighbors",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

CATEGORICAL = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
]

SOFT_TRAIN_FRACTION_RANGE = (0.20, 1.00)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]  # discrete allowed
SOFT_REVIEW_BUDGET_BOUNDS = (SOFT_REVIEW_BUDGETS[0], SOFT_REVIEW_BUDGETS[-1])  # (0.005, 0.5)


In [3]:
# ============================================================
# 3) Load dataset + surrogate
# ============================================================

df = pd.read_csv(CSV_PATH)

# Cast categoricals to str for stability (DiCE + CatBoost)
for c in CATEGORICAL:
    df[c] = df[c].astype(str)

cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

print("Loaded df:", df.shape)
print("Loaded surrogate:", MODEL_PATH)



Loaded df: (2999, 14)
Loaded surrogate: surrogate_pkl_cfs_metadata/surrogate_catboost.cbm


In [4]:
# ============================================================
# 3B) LLM setup + JSON generator
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()


def generate_greedy(inputs, max_new_tokens: int):
    # Build a fresh config each call to avoid deprecated mixing of
    # generation_config + generation kwargs and to keep pure greedy decoding.
    gen_cfg = GenerationConfig(
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return model.generate(**inputs, generation_config=gen_cfg)


print("Loaded LLM:", MODEL_ID)

@torch.inference_mode()
def llama_generate_json(user_text: str, max_new_tokens: int = 400) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "Output ONLY valid JSON. No extra text."},
            {"role": "user", "content": user_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = "SYSTEM: Output ONLY valid JSON.\nUSER:\n" + user_text + "\nASSISTANT:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = generate_greedy(inputs, max_new_tokens=max_new_tokens)

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return extract_first_json_object(decoded)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded LLM: Qwen/Qwen2.5-3B-Instruct


In [5]:
# ============================================================
# 3C) Snapping helpers (same behavior as notebook 6)
# ============================================================

def q01(x: float, lo: float, hi: float) -> float:
    """Quantize to 1% steps and clip to [lo, hi]."""
    x = max(lo, min(hi, float(x)))
    return round(x * 100.0) / 100.0

def snap_budget(x: float) -> float:
    """Snap review_budget to nearest allowed discrete value."""
    x = float(x)
    return min(SOFT_REVIEW_BUDGETS, key=lambda v: abs(v - x))



In [6]:
# ============================================================
# 3D) JSON extractor
# ============================================================

def extract_first_json_object(text: str) -> str:
    """
    Extract the first top-level JSON object {...} from model output.
    Raises ValueError if none found or braces are unbalanced.
    """
    text = text.strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError("Unbalanced JSON braces in model output.")



In [7]:
# ============================================================
# 3E) Output schema (strict contract for the LLM)
# ============================================================

QUERY_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "additionalProperties": False,
    "required": ["objective", "soft_constraints", "selection"],
    "properties": {
        "objective": {
            "type": "object",
            "additionalProperties": False,
            "required": ["type"],
            "properties": {
                "type": {"type": "string", "enum": ["target_min", "delta_improve", "target_band"]},
                "value": {"type": "number"},
                "delta": {"type": "number"},
                "lower": {"type": "number"},
                "upper": {"type": "number"},
                "eps":  {"type": "number"},
            },
        },
        "soft_constraints": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "params_soft_train_fraction",
                "params_soft_review_budget",
                "params_soft_corruption_level",
            ],
            "properties": {
                "params_soft_train_fraction": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "range"]},
                        "value": {"type": "number"},
                        "lower": {"type": "number"},
                        "upper": {"type": "number"},
                    },
                },
                "params_soft_review_budget": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "range"]},
                        "value": {"type": "number"},
                        "lower": {"type": "number"},
                        "upper": {"type": "number"},
                    },
                },
                "params_soft_corruption_level": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "allowed"]},
                        "value": {"type": "string", "enum": ["none", "mild", "strong"]},
                        "allowed": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"type": "string", "enum": SOFT_CORRUPTION_LEVELS},
                        },
                    },
                },
            },
        },
        "selection": {
            "type": "object",
            "additionalProperties": False,
            "required": ["k", "mode"],
            "properties": {
                "k": {"type": "integer", "minimum": 1, "maximum": 50},
                "mode": {"type": "string", "enum": ["best", "balanced", "closest", "diverse"]},
            },
        },
    },
}



In [8]:
# ============================================================
# 3F) Defaults + fallback + repair
# ============================================================

DEFAULT_EPS = 0.01
DEFAULT_K = 5
DEFAULT_MODE = "balanced"
MAX_RETRIES = 3

def fallback_query_core(x_factual: Dict[str, Any], y_factual_sur: float,
                        eps: float = DEFAULT_EPS,
                        k: int = DEFAULT_K,
                        mode: str = DEFAULT_MODE) -> Dict[str, Any]:
    return {
        "objective": {"type": "delta_improve", "delta": 0.02, "eps": eps},
        "soft_constraints": {
            "params_soft_train_fraction": {"mode": "fixed", "value": float(x_factual["params_soft_train_fraction"])},
            "params_soft_review_budget": {"mode": "fixed", "value": float(x_factual["params_soft_review_budget"])},
            "params_soft_corruption_level": {"mode": "fixed", "value": str(x_factual["params_soft_corruption_level"])},
        },
        "selection": {"k": int(k), "mode": str(mode)},
    }



In [9]:
# ============================================================
# 3G) Contract prompt
# ============================================================

CONTRACT_PROMPT = f"""
Return ONLY valid JSON. No markdown. No extra text.

Keys required: objective, soft_constraints, selection.

objective.type in: target_min | delta_improve | target_band

soft_constraints must contain:
- params_soft_train_fraction: mode free|fixed|range
- params_soft_review_budget:  mode free|fixed|range
- params_soft_corruption_level: mode free|fixed|allowed

selection must contain:
- k (1..50)
- mode in best|balanced|closest|diverse

Numeric constraints:
- eps > 0
- if target_band: lower <= upper
- if range: lower <= upper

Intent rules:
- If user provides explicit numeric value/range for a soft constraint, keep that value/range (do not widen to free/full range).
- If user says "best performance", set selection.mode to "best".

Allowed corruption values: {SOFT_CORRUPTION_LEVELS}
Train fraction allowed range: {SOFT_TRAIN_FRACTION_RANGE}
Review budget allowed values are multiples of 0.005 in [0.005, 0.5].

"""



In [10]:
# ============================================================
# 3H) generation + repair
# ============================================================

def repair_and_snap(query: dict, x_factual: dict, y_factual_sur: float):
    q = copy.deepcopy(query)

    obj = q.get("objective", {})
    t = obj.get("type")

    if t == "target_band":
        lo = obj.get("lower", None)
        hi = obj.get("upper", None)
        if lo is None or hi is None:
            lo, hi = float(y_factual_sur), float(y_factual_sur)

        lo, hi = float(lo), float(hi)
        if lo > hi:
            lo, hi = hi, lo

        q["objective"] = {"type": "target_band", "lower": lo, "upper": hi, "eps": float(obj.get("eps", 0.01))}

    elif t == "target_min":
        v = obj.get("value", float(y_factual_sur))
        q["objective"] = {"type": "target_min", "value": float(v), "eps": float(obj.get("eps", 0.01))}

    elif t == "delta_improve":
        d = float(obj.get("delta", 0.0))
        q["objective"] = {"type": "delta_improve", "delta": d, "eps": float(obj.get("eps", 0.01))}

    else:
        # safe fallback
        lo = hi = float(y_factual_sur)
        q["objective"] = {"type": "target_band", "lower": lo, "upper": hi, "eps": 0.01}

    # ---- selection defaults ----
    sel = q.get("selection", {})
    sel["k"] = int(sel.get("k", DEFAULT_K))
    sel["k"] = max(1, min(50, sel["k"]))
    sel["mode"] = canonical_selection_mode(sel.get("mode", DEFAULT_MODE))
    q["selection"] = sel

    # ---- soft constraints canonicalization ----
    sc = q.get("soft_constraints", {})
    out = {}

    def fallback_fixed(feat):
        return {"mode": "fixed", "value": x_factual.get(feat)}

    # review budget
    rb = sc.get("params_soft_review_budget", {"mode": "free"})
    mode = rb.get("mode", "free")
    if mode == "fixed":
        v = rb.get("value", x_factual.get("params_soft_review_budget"))
        v = float(snap_budget(_clamp(float(v), *SOFT_REVIEW_BUDGET_BOUNDS)))
        out["params_soft_review_budget"] = {"mode": "fixed", "value": v}
    elif mode == "range":
        lo = rb.get("lower", None); hi = rb.get("upper", None)
        if lo is None or hi is None:
            out["params_soft_review_budget"] = fallback_fixed("params_soft_review_budget")
        else:
            lo, hi = float(lo), float(hi)
            lo, hi = (hi, lo) if lo > hi else (lo, hi)
            lo = float(snap_budget(_clamp(lo, *SOFT_REVIEW_BUDGET_BOUNDS)))
            hi = float(snap_budget(_clamp(hi, *SOFT_REVIEW_BUDGET_BOUNDS)))
            out["params_soft_review_budget"] = {"mode": "range", "lower": lo, "upper": hi}
    elif mode == "free":
        out["params_soft_review_budget"] = {"mode": "free"}
    else:
        out["params_soft_review_budget"] = fallback_fixed("params_soft_review_budget")

    # train fraction
    tf = sc.get("params_soft_train_fraction", {"mode": "free"})
    mode = tf.get("mode", "free")
    if mode == "fixed":
        v = tf.get("value", x_factual.get("params_soft_train_fraction"))
        v = q01(float(v), *SOFT_TRAIN_FRACTION_RANGE)
        out["params_soft_train_fraction"] = {"mode": "fixed", "value": v}
    elif mode == "range":
        lo = tf.get("lower", None); hi = tf.get("upper", None)
        if lo is None or hi is None:
            out["params_soft_train_fraction"] = fallback_fixed("params_soft_train_fraction")
        else:
            lo, hi = float(lo), float(hi)
            lo, hi = (hi, lo) if lo > hi else (lo, hi)
            lo = q01(lo, *SOFT_TRAIN_FRACTION_RANGE)
            hi = q01(hi, *SOFT_TRAIN_FRACTION_RANGE)
            out["params_soft_train_fraction"] = {"mode": "range", "lower": lo, "upper": hi}
    elif mode == "free":
        out["params_soft_train_fraction"] = {"mode": "free"}
    else:
        out["params_soft_train_fraction"] = fallback_fixed("params_soft_train_fraction")

    # corruption level (categorical)
    cl = sc.get("params_soft_corruption_level", {"mode": "free"})
    mode = cl.get("mode", "free")
    allowed_set = set(SOFT_CORRUPTION_LEVELS)
    if mode == "fixed":
        v = str(cl.get("value", x_factual.get("params_soft_corruption_level")))
        if v not in allowed_set:
            v = str(x_factual.get("params_soft_corruption_level", "none"))
            if v not in allowed_set:
                v = "none"
        out["params_soft_corruption_level"] = {"mode": "fixed", "value": v}
    elif mode == "allowed":
        allowed = cl.get("allowed", None)
        if not allowed:
            out["params_soft_corruption_level"] = fallback_fixed("params_soft_corruption_level")
        else:
            vals = [str(a) for a in allowed if str(a) in allowed_set]
            vals = sorted(set(vals))
            if vals:
                out["params_soft_corruption_level"] = {"mode": "allowed", "allowed": vals}
            else:
                out["params_soft_corruption_level"] = fallback_fixed("params_soft_corruption_level")
    elif mode == "free":
        out["params_soft_corruption_level"] = {"mode": "free"}
    else:
        out["params_soft_corruption_level"] = fallback_fixed("params_soft_corruption_level")

    q["soft_constraints"] = out
    return q

def _clamp(x, lo, hi):
    return max(lo, min(hi, x))



In [11]:
# ============================================================
# 3I) Normalization layer (coerce shorthand into canonical schema)
# ============================================================

REQUIRED_SOFT_KEYS = {
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
}

def normalize_objective(obj: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(obj, dict):
        raise ValueError("objective must be an object/dict")

    obj = dict(obj)
    t = obj.get("type", "delta_improve")

    if t == "target_min" and "value" not in obj and "target_min" in obj:
        obj["value"] = obj.pop("target_min")
        return obj

    if t == "delta_improve" and "delta" not in obj and "value" in obj:
        obj["delta"] = obj.pop("value")
        return obj

    if t == "target_band" and "lower" not in obj and "upper" not in obj and "value" in obj:
        val = obj.pop("value")
        if isinstance(val, (list, tuple)) and len(val) == 2:
            obj["lower"], obj["upper"] = val
        return obj

    if t == "target_band" and "target_band" in obj:
        val = obj.pop("target_band")
        if isinstance(val, (list, tuple)) and len(val) == 2:
            obj["lower"], obj["upper"] = val
        return obj

    return obj

def normalize_soft_constraints(sc: Dict[str, Any], x_factual: Dict[str, Any]) -> Dict[str, Any]:
    """
    Handles:
    - canonical format
    - wrapper {mode, values/value}
    - flat values
    - shorthand format
    """
    if not isinstance(sc, dict):
        raise ValueError("soft_constraints must be an object/dict")

    # Already canonical
    if REQUIRED_SOFT_KEYS.issubset(sc.keys()) and all(isinstance(sc[k], dict) for k in REQUIRED_SOFT_KEYS):
        return {k: sc[k] for k in REQUIRED_SOFT_KEYS}

    # Wrapper {"mode":..., "values"/"value": {...}}
    if "mode" in sc and ("values" in sc or "value" in sc):
        vals = sc.get("values", sc.get("value"))
        if isinstance(vals, dict):
            if REQUIRED_SOFT_KEYS.issubset(vals.keys()):
                return normalize_soft_constraints(vals, x_factual=x_factual)
            sc = vals

    out = {}

    # 1) train fraction
    tf_mode = sc.get("params_soft_train_fraction")
    if isinstance(tf_mode, str) and tf_mode in {"free", "fixed", "range"}:
        if tf_mode == "free":
            out["params_soft_train_fraction"] = {"mode": "free"}
        elif tf_mode == "fixed":
            v = sc.get("train_fraction", sc.get("params_soft_train_fraction_value", sc.get("value")))
            if v is None:
                v = x_factual["params_soft_train_fraction"]
            out["params_soft_train_fraction"] = {"mode": "fixed", "value": v}
        else:
            lo = sc.get("train_fraction_min", sc.get("min", sc.get("lower")))
            hi = sc.get("train_fraction_max", sc.get("max", sc.get("upper")))
            if lo is None or hi is None:
                lo, hi = SOFT_TRAIN_FRACTION_RANGE
            out["params_soft_train_fraction"] = {"mode": "range", "lower": lo, "upper": hi}
    elif isinstance(sc.get("params_soft_train_fraction"), dict):
        out["params_soft_train_fraction"] = sc["params_soft_train_fraction"]

    # 2) review budget
    rb_mode = sc.get("params_soft_review_budget")
    if isinstance(rb_mode, str) and rb_mode in {"free", "fixed", "range"}:
        if rb_mode == "free":
            out["params_soft_review_budget"] = {"mode": "free"}
        elif rb_mode == "fixed":
            v = sc.get("review_budget", sc.get("params_soft_review_budget_value", sc.get("value")))
            if v is None:
                v = x_factual["params_soft_review_budget"]
            out["params_soft_review_budget"] = {"mode": "fixed", "value": v}
        else:
            lo = sc.get("review_budget_min", sc.get("min_budget", sc.get("lower_budget")))
            hi = sc.get("review_budget_max", sc.get("max_budget", sc.get("upper_budget")))
            if lo is None or hi is None:
                lo, hi = SOFT_REVIEW_BUDGETS[0], SOFT_REVIEW_BUDGETS[-1]
            out["params_soft_review_budget"] = {"mode": "range", "lower": lo, "upper": hi}
    elif isinstance(sc.get("params_soft_review_budget"), dict):
        out["params_soft_review_budget"] = sc["params_soft_review_budget"]

    # 3) corruption level
    cl_mode = sc.get("params_soft_corruption_level")
    if isinstance(cl_mode, str) and cl_mode in {"free", "fixed", "allowed"}:
        if cl_mode == "free":
            out["params_soft_corruption_level"] = {"mode": "free"}
        elif cl_mode == "fixed":
            v = sc.get("corruption_level", sc.get("params_soft_corruption_level_value", sc.get("value")))
            if v is None:
                v = x_factual["params_soft_corruption_level"]
            out["params_soft_corruption_level"] = {"mode": "fixed", "value": v}
        else:
            vals = sc.get("allowed", sc.get("values", sc.get("levels")))
            out["params_soft_corruption_level"] = {"mode": "allowed", "allowed": vals if vals is not None else ["none"]}
    elif isinstance(sc.get("params_soft_corruption_level"), dict):
        out["params_soft_corruption_level"] = sc["params_soft_corruption_level"]

    # Flat values fallback
    if not REQUIRED_SOFT_KEYS.issubset(out.keys()):
        flat_hits = [k for k in REQUIRED_SOFT_KEYS if k in sc and not isinstance(sc[k], dict)]
        if flat_hits:
            for k in flat_hits:
                out[k] = {"mode": "fixed", "value": sc[k]}

    missing = REQUIRED_SOFT_KEYS - set(out.keys())
    if missing:
        raise ValueError(f"soft_constraints missing keys after normalization: {sorted(missing)}")

    return out


def canonical_selection_mode(mode: Any) -> str:
    m = str(mode).strip().lower()
    if m in {"best", "highest", "max", "top", "performance", "best_performance"}:
        return "best"
    if m in {"balanced", "balance"}:
        return "balanced"
    if m in {"closest", "near", "nearest"}:
        return "closest"
    if m in {"diverse", "diversity"}:
        return "diverse"
    return DEFAULT_MODE


def normalize_selection(sel: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(sel, dict):
        sel = {}
    return {
        "k": int(sel.get("k", DEFAULT_K)),
        "mode": canonical_selection_mode(sel.get("mode", DEFAULT_MODE)),
    }


def _iter_user_lines(user_text: str):
    # Split only on newlines/semicolons; keep decimal dots intact (e.g., 0.02).
    for part in re.split(r"[\r\n;]+", user_text.lower()):
        p = part.strip()
        if p:
            yield p


def extract_explicit_preferences(user_text: str, x_factual: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Deterministically capture explicit user intent from plain text.
    Only applies when the phrase is clear (e.g., exact numeric requests).
    """
    x_factual = x_factual or {}
    prefs: Dict[str, Any] = {}
    soft = {}
    selection = {}

    objective = None

    for ln in _iter_user_lines(user_text):
        # objective: min/at-least quality or performance
        m_min = re.search(
            r"(?:min(?:imum)?\s*(?:performance|quality)?\s*(?:should\s*be|is|:)?|(?:performance|quality)\s*(?:should\s*be|is|:)?\s*at\s*least)\s*([0-9]*\.?[0-9]+)",
            ln,
        )
        if m_min:
            objective = {"type": "target_min", "value": float(m_min.group(1)), "eps": DEFAULT_EPS}
            continue

        # objective: explicit improvement delta, e.g. "+0.02"
        m_delta = re.search(
            r"(?:improv(?:e|ing|ement)|increase|raise)\D*([+\-]?[0-9]*\.?[0-9]+)",
            ln,
        )
        if m_delta and objective is None:
            d = float(m_delta.group(1))
            objective = {"type": "delta_improve", "delta": max(0.0, d), "eps": DEFAULT_EPS}

    if objective is not None:
        prefs["objective"] = objective

    for ln in _iter_user_lines(user_text):
        # train fraction constraints
        if "train fraction" in ln or "training fraction" in ln:
            m_tf_range = re.search(r"(?:between|from)\s*([0-9]*\.?[0-9]+)\s*(?:and|to|-)\s*([0-9]*\.?[0-9]+)", ln)
            if not m_tf_range:
                m_tf_range = re.search(r"([0-9]*\.?[0-9]+)\s*(?:and|to|-)\s*([0-9]*\.?[0-9]+)", ln)
            if m_tf_range:
                lo, hi = float(m_tf_range.group(1)), float(m_tf_range.group(2))
                if lo > hi:
                    lo, hi = hi, lo
                soft["params_soft_train_fraction"] = {"mode": "range", "lower": lo, "upper": hi}
            else:
                m_tf_fixed = re.search(r"(?:train(?:ing)?\s*fraction)\s*(?:should\s*be|=|is|:)?\s*([0-9]*\.?[0-9]+)", ln)
                if m_tf_fixed:
                    soft["params_soft_train_fraction"] = {"mode": "fixed", "value": float(m_tf_fixed.group(1))}

        # review budget constraints
        if "review budget" in ln:
            if "current value" in ln or "freeze" in ln:
                v_cur = x_factual.get("params_soft_review_budget", None)
                if v_cur is not None:
                    soft["params_soft_review_budget"] = {"mode": "fixed", "value": float(v_cur)}
                    continue

            m_rb_range = re.search(r"(?:between|from)\s*([0-9]*\.?[0-9]+)\s*(?:and|to|-)\s*([0-9]*\.?[0-9]+)", ln)
            if not m_rb_range:
                m_rb_range = re.search(r"([0-9]*\.?[0-9]+)\s*(?:and|to|-)\s*([0-9]*\.?[0-9]+)", ln)
            if m_rb_range:
                lo, hi = float(m_rb_range.group(1)), float(m_rb_range.group(2))
                if lo > hi:
                    lo, hi = hi, lo
                soft["params_soft_review_budget"] = {"mode": "range", "lower": lo, "upper": hi}
            else:
                m_rb_fixed = re.search(r"(?:review\s*budget)\s*(?:should\s*be|=|is|:)?\s*([0-9]*\.?[0-9]+)", ln)
                if m_rb_fixed:
                    soft["params_soft_review_budget"] = {"mode": "fixed", "value": float(m_rb_fixed.group(1))}

        # corruption constraints
        if "corruption" in ln:
            if any(w in ln for w in ["free", "any", "vary", "variable"]):
                soft["params_soft_corruption_level"] = {"mode": "free"}
            else:
                vals = [v for v in SOFT_CORRUPTION_LEVELS if re.search(rf"\b{re.escape(v)}\b", ln)]
                vals = sorted(set(vals))
                if len(vals) == 1:
                    soft["params_soft_corruption_level"] = {"mode": "fixed", "value": vals[0]}
                elif len(vals) > 1:
                    soft["params_soft_corruption_level"] = {"mode": "allowed", "allowed": vals}

        # selection.k
        m_k = re.search(r"\b(\d+)\s*(?:options?|cfs?|counterfactuals?)\b", ln)
        if m_k:
            selection["k"] = int(m_k.group(1))

        # selection.mode
        if "best performance" in ln or re.search(r"\bbest\b", ln):
            selection["mode"] = "best"
        elif "balanced" in ln:
            selection["mode"] = "balanced"
        elif "closest" in ln:
            selection["mode"] = "closest"
        elif "diverse" in ln:
            selection["mode"] = "diverse"

    if soft:
        prefs["soft_constraints"] = soft
    if selection:
        prefs["selection"] = selection

    return prefs


def apply_explicit_preferences(query: Dict[str, Any], user_text: str, x_factual: Dict[str, Any] = None) -> Dict[str, Any]:
    q = copy.deepcopy(query)
    prefs = extract_explicit_preferences(user_text, x_factual=x_factual)

    if "objective" in prefs:
        q["objective"] = prefs["objective"]

    if "soft_constraints" in prefs:
        q.setdefault("soft_constraints", {})
        for k, v in prefs["soft_constraints"].items():
            q["soft_constraints"][k] = v

    if "selection" in prefs:
        q.setdefault("selection", {})
        if "k" in prefs["selection"]:
            q["selection"]["k"] = int(prefs["selection"]["k"])
        if "mode" in prefs["selection"]:
            q["selection"]["mode"] = canonical_selection_mode(prefs["selection"]["mode"])

    return q




In [12]:
# ============================================================
# 3J) Build a safe query core (normalize -> validate -> repair -> validate)
# ============================================================

def build_query_core(user_text: str,
                     x_factual: Dict[str, Any],
                     y_factual_sur: float,
                     max_retries: int = MAX_RETRIES) -> Dict[str, Any]:
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            prompt = CONTRACT_PROMPT + "\n\nUSER REQUEST:\n" + user_text.strip()
            raw = llama_generate_json(prompt)
            draft = json.loads(raw)

            core = dict(draft)
            if "objective" in core:
                core["objective"] = normalize_objective(core["objective"])
            if "soft_constraints" in core:
                core["soft_constraints"] = normalize_soft_constraints(core["soft_constraints"], x_factual=x_factual)
            if "selection" in core:
                core["selection"] = normalize_selection(core["selection"])

            validate(instance=core, schema=QUERY_SCHEMA)
            repaired = repair_and_snap(core, x_factual=x_factual, y_factual_sur=y_factual_sur)
            repaired = apply_explicit_preferences(repaired, user_text=user_text, x_factual=x_factual)
            repaired = repair_and_snap(repaired, x_factual=x_factual, y_factual_sur=y_factual_sur)
            validate(instance=repaired, schema=QUERY_SCHEMA)
            repaired["_meta"] = {"used_fallback": False, "attempts": attempt + 1}
            return repaired

        except Exception as e:
            last_error = e

    core = fallback_query_core(x_factual=x_factual, y_factual_sur=y_factual_sur)
    core = apply_explicit_preferences(core, user_text=user_text, x_factual=x_factual)
    core = repair_and_snap(core, x_factual=x_factual, y_factual_sur=y_factual_sur)
    core["_meta"] = {"used_fallback": True, "reason": str(last_error)[:250]}
    return core


In [13]:
#### ============================================================
# 3K) Prepare factual candidates + interactive user inputs
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# ---- 1) deterministic candidate pool ----
rng = np.random.default_rng(42)
idxs = rng.choice(df.index.to_numpy(), size=3, replace=False)

preview_rows = []
for j, idx in enumerate(idxs):
    x = df.loc[idx, FEATURES].to_dict()
    y = float(cb.predict(pd.DataFrame([x]))[0])
    row = {"candidate_pick": j, "df_index": int(idx), "value_surrogate": y}
    for f in FEATURES[:6]:
        row[f] = x[f]
    preview_rows.append(row)

factual_candidates = pd.DataFrame(preview_rows)
display(factual_candidates)

# ---- 2) UI inputs ----
candidate_dd = widgets.Dropdown(
    options=[(f"candidate {int(r.candidate_pick)} | df_index={int(r.df_index)} | sur={r.value_surrogate:.4f}", int(r.candidate_pick))
             for _, r in factual_candidates.iterrows()],
    value=0,
    description="Factual:",
    layout=widgets.Layout(width="900px")
)

user_text = widgets.Textarea(
    value=(
        "Improve the surrogate quality by at least +0.02.\n"
        "Freeze review budget to the current value.\n"
        "Allow corruption level to vary.\n"
        "Keep train fraction between 0.30 and 0.60.\n"
        "Return 7 options, balanced."
    ),
    description="Prefs:",
    layout=widgets.Layout(width="900px", height="140px")
)

run_btn = widgets.Button(description="Build query", button_style="success")
out = widgets.Output()

def on_build_clicked(_):
    with out:
        clear_output()

        CANDIDATE_PICK = int(candidate_dd.value)
        USER_TEXT = user_text.value.strip()

        selected_idx = int(idxs[CANDIDATE_PICK])
        x_factual = df.loc[selected_idx, FEATURES].to_dict()
        x_factual_df = pd.DataFrame([x_factual])
        y_factual_sur = float(cb.predict(x_factual_df)[0])

        print("Selected candidate_pick:", CANDIDATE_PICK)
        print("Selected df_index:", selected_idx)
        print("Factual surrogate value:", y_factual_sur)
        print("\nUSER_TEXT:\n", USER_TEXT)

        # 1) draft from LLM
        core = build_query_core(
            user_text=USER_TEXT,
            x_factual=x_factual,
            y_factual_sur=y_factual_sur
        )

        query_draft = {
            "factual": {"index": selected_idx, "x": dict(x_factual), "value_surrogate": y_factual_sur},
            "objective": core["objective"],
            "soft_constraints": core["soft_constraints"],
            "selection": core["selection"],
            "_meta": core.get("_meta", {}),
        }

        # 2) final safety pass after assembly
        query_final = repair_and_snap(query_draft, x_factual=x_factual, y_factual_sur=y_factual_sur)
        query_final["_meta"] = core.get("_meta", {})
        query_final["_user_request_text"] = USER_TEXT

        # 3) expose to downstream cells
        global query
        query = query_final
        globals()["x_factual_df"] = x_factual_df

        print("\nQUERY (FINAL, LLM-extracted):\n", json.dumps(query, indent=2))

run_btn.on_click(on_build_clicked)

display(widgets.VBox([candidate_dd, user_text, run_btn, out]))



,candidate_pick,df_index,value_surrogate,params_backbone,params_batch_size,params_center_crop_key,params_coreset_sampling_ratio,params_image_size_key,params_layers_key
0,0,1963,0.831854,resnet34,16,0.875,0.095,256,l3
1,1,267,0.186287,resnet34,16,0.875,0.087,320,l2
2,2,2320,0.890574,resnet34,16,0.875,0.099,224,l3


In [14]:
# ============================================================
# 6) Translate objective -> desired_range
# ============================================================

obj = query["objective"]
factual_base = float(query["factual"]["value_surrogate"])  # baseline for delta

if obj["type"] == "target_min":
    L, U = float(obj["value"]), 1.0
elif obj["type"] == "delta_improve":
    L, U = factual_base + float(obj["delta"]), 1.0
elif obj["type"] == "target_band":
    L, U = float(obj["lower"]), float(obj["upper"])
else:
    raise ValueError(f"Unknown objective type: {obj['type']}")

desired_range = [max(0.0, L), min(1.0, U)]
print("desired_range:", desired_range)



desired_range: [0.9105744295099764, 1.0]


In [15]:
# ============================================================
# 7) DiCE setup + permitted_range (Option C) + generate CFs
# ============================================================

# DiCE training dataframe
df_dice = df[FEATURES + [TARGET]].copy()

CONTINUOUS = [c for c in FEATURES if c not in CATEGORICAL]
data_dice = dice_ml.Data(dataframe=df_dice, continuous_features=CONTINUOUS, outcome_name=TARGET)
model_dice = dice_ml.Model(model=cb, backend="sklearn", model_type="regressor")
exp_dice = Dice(data_dice, model_dice, method="random")

# factual to DiCE
x0_dice = x_factual_df[FEATURES].copy()

def domain_for_feature(feat: str):
    if feat in CATEGORICAL:
        return sorted(df_dice[feat].dropna().astype(str).unique().tolist())
    col = pd.to_numeric(df_dice[feat], errors="coerce")
    return [float(np.nanmin(col)), float(np.nanmax(col))]

def factual_value(feat: str):
    v = x0_dice.iloc[0][feat]
    return str(v) if feat in CATEGORICAL else float(v)

def safe_range(lo, hi, feat):
    try:
        lo = float(lo); hi = float(hi)
        if lo > hi:
            return None
        dom_lo, dom_hi = domain_for_feature(feat)
        lo = max(dom_lo, lo)
        hi = min(dom_hi, hi)
        if lo > hi:
            return None
        return [float(lo), float(hi)]
    except Exception:
        return None

def safe_allowed(allowed, feat):
    try:
        dom = set(map(str, domain_for_feature(feat)))
        allowed = [str(a) for a in allowed]
        allowed = [a for a in allowed if a in dom]
        return sorted(set(allowed)) if allowed else None
    except Exception:
        return None

# Build permitted_range from soft_constraints.
# If invalid/missing -> default to factual fixed.
permitted_range = {}
for feat, spec in query["soft_constraints"].items():
    mode = spec.get("mode", "free")

    if mode == "free":
        permitted_range[feat] = domain_for_feature(feat)

    elif mode == "fixed":
        if feat in CATEGORICAL:
            v = str(spec.get("value", factual_value(feat)))
            ok = safe_allowed([v], feat) or safe_allowed([factual_value(feat)], feat)
            permitted_range[feat] = ok
        else:
            v = spec.get("value", factual_value(feat))
            r = safe_range(v, v, feat) or safe_range(factual_value(feat), factual_value(feat), feat)
            permitted_range[feat] = r

    elif mode == "range":
        if feat in CATEGORICAL:
            # range not supported for categoricals -> factual fixed
            permitted_range[feat] = safe_allowed([factual_value(feat)], feat)
        else:
            lo = spec.get("lower", None)
            hi = spec.get("upper", None)
            if lo is None or hi is None:
                permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)
            else:
                r = safe_range(lo, hi, feat)
                permitted_range[feat] = r if r is not None else safe_range(factual_value(feat), factual_value(feat), feat)

    elif mode == "allowed":
        if feat not in CATEGORICAL:
            permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)
        else:
            allowed = spec.get("allowed", None)
            ok = safe_allowed(allowed, feat) if allowed is not None else None
            permitted_range[feat] = ok if ok is not None else safe_allowed([factual_value(feat)], feat)

    else:
        # unknown mode -> factual fixed
        if feat in CATEGORICAL:
            permitted_range[feat] = safe_allowed([factual_value(feat)], feat)
        else:
            permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)

# Remove None entries
permitted_range = {k: v for k, v in permitted_range.items() if v is not None}

print("\npermitted_range (constraints injected into DiCE search):")
for k_, v_ in permitted_range.items():
    print(f" - {k_}: {v_}")

pool = int(max(20, 5 * query["selection"]["k"]))
print("\nGenerating pool:", pool)
print("desired_range:", desired_range)

dice_res = exp_dice.generate_counterfactuals(
    x0_dice,
    total_CFs=pool,
    desired_range=desired_range,
    features_to_vary=FEATURES,
    permitted_range=permitted_range,
)




permitted_range (constraints injected into DiCE search):
 - params_soft_review_budget: [0.365, 0.365]
 - params_soft_train_fraction: [0.3, 1.0]
 - params_soft_corruption_level: ['mild', 'none', 'strong']

Generating pool: 35
desired_range: [0.9105744295099764, 1.0]


100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


In [19]:
# ============================================================
# 8) Final CF table: minimal safety band + rank + top-k
# ============================================================

ex = dice_res.cf_examples_list[0]
cf_raw = ex.final_cfs_df_sparse.copy() if hasattr(ex, "final_cfs_df_sparse") else ex.final_cfs_df.copy()

assert TARGET in cf_raw.columns, f"DiCE CF table missing '{TARGET}'. Columns: {list(cf_raw.columns)}"

cf_df = cf_raw[FEATURES + [TARGET]].copy()
for c in CATEGORICAL:
    if c in cf_df.columns:
        cf_df[c] = cf_df[c].astype(str)

print(f"CFs before desired_range filter: {len(cf_df)}")

# safety: enforce desired band
L, U = desired_range
cf_df = cf_df[(cf_df[TARGET] >= L) & (cf_df[TARGET] <= U)].copy()

print(f"CFs after desired_range filter [{L:.6f}, {U:.6f}]: {len(cf_df)}")

# rank and take top-k (using DiCE 'value' only)
k = int(query["selection"]["k"])
cf_df = cf_df.sort_values(TARGET, ascending=False).reset_index(drop=True).head(k)

print(f"CFs after top-k (k={k}): {len(cf_df)}")
display(cf_df)
print(f"Kept {len(cf_df)} CFs (requested {k}).")


CFs before desired_range filter: 1
CFs after desired_range filter [0.910574, 1.000000]: 1
CFs after top-k (k=7): 1


,params_backbone,params_batch_size,params_center_crop_key,params_coreset_sampling_ratio,params_image_size_key,params_layers_key,params_max_patches_per_image,params_num_neighbors,params_pre_trained,params_reduction,params_soft_corruption_level,params_soft_review_budget,params_soft_train_fraction,value
0,resnet50,8,0.875,0.099,256,l3,1024,2,True,mean,none,0.365,0.89,0.911063


Kept 1 CFs (requested 7).


In [17]:
# ============================================================
# 9) EXPORT
# ============================================================

EXPORT_DIR = Path("surrogate_pkl_cfs_metadata")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

cfs_out = cf_df.copy()
cfs_out.insert(0, "cf_id", range(len(cfs_out)))
cfs_out = cfs_out[["cf_id"] + FEATURES + [TARGET]]
cfs_out.to_csv(EXPORT_DIR / "cfs.csv", index=False)

meta = {
    "surrogate_model_path": str(MODEL_PATH),
    "splits_dir": "patchcore_splits",
    "target": TARGET,
    "features": FEATURES,
    "categorical": CATEGORICAL,
    "factual_index": int(query["factual"]["index"]),
    "factual_value_surrogate": float(query["factual"]["value_surrogate"]),
    "objective": query["objective"],
    "soft_constraints": query["soft_constraints"],
    "selection": query["selection"],
    "desired_range": desired_range,
    "permitted_range": permitted_range,
    "n_cfs": int(len(cfs_out)),
}
with open(EXPORT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Saved:", EXPORT_DIR / "cfs.csv")
print("Saved:", EXPORT_DIR / "metadata.json")


Saved: surrogate_pkl_cfs_metadata/cfs.csv
Saved: surrogate_pkl_cfs_metadata/metadata.json


In [22]:
# ============================================================
# 10) LLM explanation of CFs (alignment + selection guidance)
# ============================================================

if cf_df.empty:
    print("No counterfactuals available to explain.")
else:
    factual_x = query["factual"]["x"]
    factual_sur = float(query["factual"]["value_surrogate"])
    user_request_text = query.get("_user_request_text", "")

    def to_py(v):
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.bool_,)):
            return bool(v)
        return v

    def objective_ok(score: float, objective: Dict[str, Any], factual_value: float) -> bool:
        t = objective.get("type")
        if t == "target_min":
            return float(score) >= float(objective.get("value", factual_value))
        if t == "delta_improve":
            return float(score) >= float(factual_value) + float(objective.get("delta", 0.0))
        if t == "target_band":
            lo = float(objective.get("lower", 0.0))
            hi = float(objective.get("upper", 1.0))
            return lo <= float(score) <= hi
        return False

    def soft_ok(row: pd.Series, feat: str, spec: Dict[str, Any]):
        mode = spec.get("mode", "free")
        val = row.get(feat)

        if mode == "free":
            return True, "free"

        if mode == "fixed":
            target = spec.get("value")
            if feat in CATEGORICAL:
                ok = str(val) == str(target)
            else:
                ok = abs(float(val) - float(target)) <= 1e-12
            return ok, f"fixed={target}"

        if mode == "range":
            lo = spec.get("lower", None)
            hi = spec.get("upper", None)
            if lo is None or hi is None:
                return False, "range missing bounds"
            v = float(val)
            ok = float(lo) <= v <= float(hi)
            return ok, f"range=[{float(lo)}, {float(hi)}]"

        if mode == "allowed":
            allowed = [str(a) for a in spec.get("allowed", [])]
            ok = str(val) in set(allowed)
            return ok, f"allowed={allowed}"

        return False, f"unknown mode={mode}"

    def major_changes(row: pd.Series, max_items: int = 4):
        diffs = []
        for feat in FEATURES:
            if feat not in row:
                continue
            cf_val = row[feat]
            fx_val = factual_x[feat]
            if feat in CATEGORICAL:
                if str(cf_val) != str(fx_val):
                    diffs.append((1.0, f"{feat}: {fx_val} -> {cf_val}"))
            else:
                d = float(cf_val) - float(fx_val)
                if abs(d) > 1e-12:
                    diffs.append((abs(d), f"{feat}: {float(fx_val):.4g} -> {float(cf_val):.4g} ({d:+.4g})"))
        diffs.sort(key=lambda x: x[0], reverse=True)
        return [txt for _, txt in diffs[:max_items]], len(diffs)

    eval_source = cf_df.copy().reset_index(drop=True)
    eval_source.insert(0, "cf_id", range(len(eval_source)))
    
    records = []
    for _, row in eval_source.iterrows():
        score = float(row[TARGET])
        obj_match = objective_ok(score, query["objective"], factual_sur)
    
        checks = {}
        violated = []
        for feat, spec in query["soft_constraints"].items():
            ok, rule = soft_ok(row, feat, spec)
            checks[feat] = {
                "ok": bool(ok),
                "rule": rule,
                "value": to_py(row.get(feat)),
            }
            if not ok:
                violated.append(feat)
    
        changes_top, n_changes = major_changes(row)
        n_soft = max(1, len(query["soft_constraints"]))
        soft_ok_count = sum(1 for v in checks.values() if v["ok"])
    
        records.append({
            "cf_id": int(row["cf_id"]),
            "score": score,
            "objective_match": bool(obj_match),
            "soft_ok_count": int(soft_ok_count),
            "soft_total": int(n_soft),
            "soft_match_rate": float(soft_ok_count / n_soft),
            "violated_constraints": violated,
            "changed_feature_count": int(n_changes),
            "top_changes": changes_top,
            "checks": checks,
        })
    
    eval_df = pd.DataFrame([
        {
            "cf_id": r["cf_id"],
            "score": r["score"],
            "objective_match": r["objective_match"],
            "soft_match_rate": r["soft_match_rate"],
            "violations": ", ".join(r["violated_constraints"]) if r["violated_constraints"] else "none",
            "changed_feature_count": r["changed_feature_count"],
        }
        for r in records
    ]).sort_values(["objective_match", "soft_match_rate", "score"], ascending=[False, False, False]).reset_index(drop=True)
    
    print("CF alignment table:")
    display(eval_df)
    
    def grounded_summary(eval_df: pd.DataFrame, records: list) -> str:
        lines = []
        by_id = {r["cf_id"]: r for r in records}
        for _, row in eval_df.iterrows():
            rid = int(row["cf_id"])
            r = by_id[rid]
            lines.append(
                f"CF {rid}: score={r['score']:.6f}, objective_match={r['objective_match']}, "
                f"soft_match_rate={r['soft_match_rate']:.2f}, violations="
                f"{', '.join(r['violated_constraints']) if r['violated_constraints'] else 'none'}, "
                f"changed_features={r['changed_feature_count']}."
            )
        best_id = int(eval_df.iloc[0]["cf_id"])
        lines.append(f"Best choice: CF {best_id}.")
        if len(eval_df) > 1:
            runner_id = int(eval_df.iloc[1]["cf_id"])
            lines.append(f"Runner-up: CF {runner_id}.")
        else:
            lines.append("Runner-up: none (only one counterfactual available).")
        return "\n".join(lines)
    
    payload = {
        "user_request_text": user_request_text,
        "parsed_query": {
            "objective": query["objective"],
            "soft_constraints": query["soft_constraints"],
            "selection": query["selection"],
        },
        "factual": {
            "index": int(query["factual"]["index"]),
            "value_surrogate": float(query["factual"]["value_surrogate"]),
        },
        "counterfactuals": records,
    }
    
    valid_ids = [int(r["cf_id"]) for r in records]
    valid_ids_set = set(valid_ids)
    
    payload = {
        "user_request_text": user_request_text,
        "parsed_query": {
            "objective": query["objective"],
            "soft_constraints": query["soft_constraints"],
            "selection": query["selection"],
        },
        "factual": {
            "index": int(query["factual"]["index"]),
            "value_surrogate": float(query["factual"]["value_surrogate"]),
        },
        "counterfactuals": records,
    }
    
    def _parse_json_obj(text: str) -> Dict[str, Any]:
        t = (text or "").strip()
    
        # If model wraps output in ```json ... ```
        fence = re.search(r"```(?:json)?\s*(.*?)\s*```", t, flags=re.IGNORECASE | re.DOTALL)
        if fence:
            t = fence.group(1).strip()
    
        dec = json.JSONDecoder()
        for i, ch in enumerate(t):
            if ch != "{":
                continue
            try:
                obj, _ = dec.raw_decode(t[i:])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                pass
    
        raise ValueError("No valid JSON object found in model output.")
    
    
    def _validate_llm_json(obj: Dict[str, Any], valid_ids: list[int]) -> Dict[str, Any]:
        valid_set = set(valid_ids)
    
        if not isinstance(obj, dict):
            raise ValueError("Root must be a JSON object.")
    
        per_cf = obj.get("per_cf")
        if not isinstance(per_cf, list):
            raise ValueError("`per_cf` must be a list.")
    
        seen = []
        for item in per_cf:
            if not isinstance(item, dict):
                raise ValueError("Each `per_cf` item must be an object.")
    
            if "cf_id" not in item:
                raise ValueError("Each `per_cf` item must include `cf_id`.")
            cid = int(item["cf_id"])
            if cid not in valid_set:
                raise ValueError(f"Invalid cf_id in per_cf: {cid}")
    
            bullets = item.get("bullets")
            if not isinstance(bullets, list) or not (2 <= len(bullets) <= 3):
                raise ValueError(f"CF {cid}: `bullets` must be a list of length 2-3.")
            if any(not isinstance(b, str) or not b.strip() for b in bullets):
                raise ValueError(f"CF {cid}: each bullet must be a non-empty string.")
    
            seen.append(cid)
    
        if len(seen) != len(set(seen)):
            raise ValueError("Duplicate cf_id entries in per_cf.")
        if set(seen) != valid_set:
            raise ValueError(f"per_cf IDs mismatch. expected={sorted(valid_set)} got={sorted(set(seen))}")
    
        best = obj.get("best_choice")
        if not isinstance(best, dict):
            raise ValueError("`best_choice` must be an object.")
        if "cf_id" not in best:
            raise ValueError("`best_choice.cf_id` is required.")
        best_id = int(best["cf_id"])
        if best_id not in valid_set:
            raise ValueError(f"Invalid best_choice cf_id: {best_id}")
    
        runner = obj.get("runner_up")
        if not isinstance(runner, dict):
            raise ValueError("`runner_up` must be an object.")
        runner_id = runner.get("cf_id", None)
    
        if len(valid_ids) == 1:
            if runner_id is not None:
                raise ValueError("runner_up.cf_id must be null when only one CF exists.")
        else:
            if runner_id is None:
                raise ValueError("runner_up.cf_id must be non-null when >=2 CFs exist.")
            runner_id = int(runner_id)
            if runner_id not in valid_set:
                raise ValueError(f"Invalid runner_up cf_id: {runner_id}")
            if runner_id == best_id:
                raise ValueError("runner_up.cf_id must differ from best_choice.cf_id.")
    
        return obj
    
    
    def llm_explain_json(payload: Dict[str, Any], valid_ids: list[int], max_retries: int = 8) -> Dict[str, Any]:
        base_prompt = (
            "You are helping a user choose among generated counterfactuals.\n"
            "Use only the provided structured data.\n"
            f"Valid CF IDs are exactly: {valid_ids}. Never mention any other ID.\n"
            "Return ONLY valid JSON with this exact schema:\n"
            "{\n"
            '  "per_cf": [\n'
            '    {"cf_id": <int>, "bullets": ["...", "...", "..."]}\n'
            "  ],\n"
            '  "best_choice": {"cf_id": <int>, "why": "<one concise paragraph>"},\n'
            '  "runner_up": {"cf_id": <int or null>, "why": "<one concise paragraph>"}\n'
            "}\n"
            "Rules:\n"
            "- Include every CF exactly once in per_cf.\n"
            "- Each CF must have 2-3 bullets.\n"
            "- If only one CF exists, runner_up.cf_id must be null.\n"
            "- Do not invent values.\n\n"
            f"DATA:\n{json.dumps(payload, indent=2)}\n\n"
            "JSON:\n"
        )
    
        last_err = ""
        last_raw = ""
    
        for attempt in range(1, max_retries + 1):
            prompt = base_prompt
            if last_err:
                prompt += (
                    f"\nPrevious output failed validation: {last_err}\n"
                    "Regenerate valid JSON only.\n"
                )
                if last_raw:
                    prompt += f"\nInvalid output was:\n{last_raw[:1200]}\n"
    
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.inference_mode():
                out = generate_greedy(inputs, max_new_tokens=700)
    
            gen_ids = out[0][inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            last_raw = raw
    
            try:
                obj = _parse_json_obj(raw)
                return _validate_llm_json(obj, valid_ids)
            except Exception as e:
                last_err = str(e)
    
        raise RuntimeError(
            f"LLM failed to produce valid explanation JSON after {max_retries} attempts. Last error: {last_err}"
        )

    
    resp = llm_explain_json(payload, valid_ids, max_retries=6)
    
    order = {cid: i for i, cid in enumerate(valid_ids)}
    per_cf_sorted = sorted(resp["per_cf"], key=lambda x: order[int(x["cf_id"])])
    
    lines = []
    for item in per_cf_sorted:
        cid = int(item["cf_id"])
        lines.append(f"CF {cid}:")
        for b in item["bullets"][:3]:
            lines.append(f"- {str(b).strip()}")
    
    best_id = int(resp["best_choice"]["cf_id"])
    lines.append(f"Best choice: CF {best_id}. {str(resp['best_choice'].get('why', '')).strip()}")
    
    runner_id = resp.get("runner_up", {}).get("cf_id", None)
    runner_why = str(resp.get("runner_up", {}).get("why", "")).strip()
    if runner_id is None:
        lines.append(f"Runner-up: none. {runner_why}")
    else:
        lines.append(f"Runner-up: CF {int(runner_id)}. {runner_why}")
    
    explanation = "\n".join(lines)
    
    print("\nLLM explanation and recommendation:\n")
    print(explanation)
    
    explain_path = EXPORT_DIR / "cf_explanations.txt"
    with open(explain_path, "w", encoding="utf-8") as f:
        f.write(explanation + "\n")
    print("\nSaved:", explain_path)



CF alignment table:


,cf_id,score,objective_match,soft_match_rate,violations,changed_feature_count
0,0,0.911063,True,1.0,none,3



LLM explanation and recommendation:

CF 0:
- More complex features can lead to higher accuracy and thus a higher surrogate quality score.
- Batch size reduction from 16 to 8 helps in reducing overfitting and improving generalization.
- Image size increase from 224 to 256 provides more context and detail in the input images.
Best choice: CF 0. The best choice maintains the original surrogate quality score while addressing the constraints. It freezes the review budget and sets the corruption level to 'none', ensuring a balanced approach.
Runner-up: none. There is no alternative counterfactual that significantly improves the surrogate quality score without violating constraints.

Saved: surrogate_pkl_cfs_metadata/cf_explanations.txt
